# Week 4: HITL — Adding Human Review to the Pipeline

Last week we built: **Extract → Retrieve → Classify**

This week we add the missing pieces:
- **Validator gate** — pause on Critical severity or low confidence for human approval
- **Override** — change the classification after the fact
- **Trainer log** — capture human corrections for system improvement

**What you'll learn:**
- LangGraph checkpointers — saving state so you can pause and resume
- `interrupt()` — conditional pauses for human review
- `update_state()` — modifying pipeline state from outside the graph
- The 3 HITL patterns: Trainer, Validator, Override

## Setup

In [ ]:
import json
import warnings
from typing import Optional
from enum import Enum
from datetime import datetime

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

load_dotenv()

True

## Part 1: Rebuild the Week 3 Pipeline (Quick)

Same extract → retrieve → classify pipeline from last week. If you've done Week 3, this is all familiar.

In [ ]:
# --- Load data + build vector store (same as Week 3) ---

with open("problem_codes.json") as f:
    problem_codes = json.load(f)

documents = []
for pc in problem_codes:
    text = f"{pc['code']}: {pc['category']} — {pc['subcategory']}\n{pc['description']}\nKeywords: {', '.join(pc['keywords'])}"
    doc = Document(page_content=text, metadata={"code": pc["code"], "category": pc["category"]})
    documents.append(doc)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents, embeddings, collection_name="problem_codes")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vector store ready with {len(documents)} problem codes")

Vector store ready with 21 problem codes


In [ ]:
# --- Pydantic models (same as Week 3) ---

class Severity(str, Enum):
    CRITICAL = "Critical"
    HIGH = "High"
    MEDIUM = "Medium"
    LOW = "Low"


class MaintenanceEntity(BaseModel):
    problem_type: str = Field(description="Brief description of the maintenance problem")
    location_building: str = Field(description="Name of the building")
    location_detail: Optional[str] = Field(description="Specific location within the building")
    severity: Severity = Field(description="Critical / High / Medium / Low")
    caller_role: Optional[str] = Field(description="Role of the caller")
    urgency_indicators: list[str] = Field(description="Phrases indicating urgency")
    summary: str = Field(description="One-sentence summary")


class Classification(BaseModel):
    selected_code: str = Field(description="The problem code that best matches (e.g., PLUMB-001)")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    reasoning: str = Field(description="Brief explanation of why this code was selected")


# --- Extended state: adds routing_decision and trainer_log for HITL ---

class PipelineState(TypedDict):
    transcript: str
    entities: Optional[dict]
    retrieved_codes: Optional[list[dict]]
    classification: Optional[dict]
    routing_decision: Optional[str]         # NEW: what happened after classification
    trainer_log: Optional[list[dict]]       # NEW: human corrections captured for learning

In [ ]:
# --- Nodes 1-3: same as Week 3 (extract, retrieve, classify) ---

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

EXTRACTION_PROMPT = """
You are an expert building maintenance call analyst. Extract key information 
from this transcript.

Severity guidelines:
- Critical: Immediate danger to life/safety (gas leak, fire, trapped persons, flooding)
- High: Significant disruption or escalation risk (major leak, broken security glass, HVAC failure)
- Medium: Needs attention soon, not emergency (broken door, minor plumbing, elevator malfunction)
- Low: Minor/cosmetic (flickering light, carpet stain, empty soap dispenser)

Transcript:
{transcript}
"""

CLASSIFICATION_PROMPT = """
You are classifying a building maintenance issue. Based on the extracted 
information and the candidate problem codes retrieved from our database, 
select the best matching code.

Extracted information:
- Problem: {problem_type}
- Severity: {severity}
- Summary: {summary}

Candidate problem codes:
{candidates}

Respond with your classification.
"""


def extract_entities(state: PipelineState) -> dict:
    """Node 1: Extract structured entities from the raw transcript."""
    extractor = llm.with_structured_output(MaintenanceEntity)
    result = extractor.invoke(EXTRACTION_PROMPT.format(transcript=state["transcript"]))
    return {"entities": result.model_dump()}


def retrieve_codes(state: PipelineState) -> dict:
    """Node 2: RAG retrieval — find the top-k matching problem codes."""
    entities = state["entities"]
    query = f"{entities['problem_type']} {entities['summary']}"
    docs = retriever.invoke(query)
    codes = [{"code": d.metadata["code"], "content": d.page_content} for d in docs]
    return {"retrieved_codes": codes}


def classify_problem(state: PipelineState) -> dict:
    """Node 3: Classify using extracted entities + retrieved codes."""
    entities = state["entities"]
    codes = state["retrieved_codes"]
    candidates = "\n\n".join(c["content"] for c in codes)

    classifier = llm.with_structured_output(Classification)
    result = classifier.invoke(CLASSIFICATION_PROMPT.format(
        problem_type=entities["problem_type"],
        severity=entities["severity"],
        summary=entities["summary"],
        candidates=candidates,
    ))
    return {"classification": result.model_dump()}


print("Nodes 1-3 defined (same as Week 3)")

Nodes 1-3 defined (same as Week 3)


## Part 2: The New Nodes — HITL

Here's what we're adding to the Week 3 pipeline:

```
[1. Extract] → [2. Retrieve] → [3. Classify] → [4. Review & Route] → [5. Log Result] → END
                                                       ↑
                                                 NEW: checks severity
                                                 + confidence, pauses
                                                 if Critical or < 0.7
```

This maps to the full pipeline from the slides:
- Steps 1-3: same as Week 3 (extract, retrieve, classify)
- Step 4 (`review_and_route`): combines **confidence check** + **human review gate** + **routing decision** into one node. Checks severity and confidence — if Critical or confidence < 0.7, pauses via `interrupt()` for human approval. Otherwise, auto-routes.
- Step 5 (`log_result`): the **trainer pattern** — logs every decision (auto-routed, approved, overridden) for future system improvement.

In [ ]:
def review_and_route(state: PipelineState) -> dict:
    """Node 4: Validator gate — conditionally pauses for human review.
    
    If severity is Critical OR confidence < 0.7:
        → interrupt() pauses the pipeline
        → human reviews and either approves or provides an override
        → pipeline resumes with the human's decision
    
    Otherwise:
        → auto-route (no human needed)
    
    This is the VALIDATOR pattern: the human approves BEFORE the action happens.
    """
    classification = state["classification"]
    entities = state["entities"]

    severity = entities.get("severity", "Medium")
    # Handle both "Critical" and "Severity.CRITICAL" formats from Pydantic
    severity_str = severity.split(".")[-1] if "." in str(severity) else str(severity)
    confidence = classification.get("confidence", 0.5)

    needs_review = severity_str == "Critical" or confidence < 0.7

    if needs_review:
        reason = []
        if severity_str == "Critical":
            reason.append("severity is Critical")
        if confidence < 0.7:
            reason.append(f"confidence is low ({confidence:.2f})")

        print(f"\n{'='*60}")
        print(f"  HUMAN REVIEW REQUIRED — {' and '.join(reason)}")
        print(f"  Code:       {classification['selected_code']}")
        print(f"  Confidence: {confidence:.2f}")
        print(f"  Severity:   {severity_str}")
        print(f"  Reasoning:  {classification['reasoning']}")
        print(f"{'='*60}")

        # interrupt() PAUSES the pipeline here.
        # It returns whatever value the human passes via Command(resume=...).
        # The pipeline won't proceed past this line until resumed.
        decision = interrupt({
            "type": "review_required",
            "reason": " and ".join(reason),
            "severity": severity_str,
            "classification": classification,
        })

        # --- Handle the human's decision ---
        if decision.get("override_code"):
            # OVERRIDE: human changed the classification
            new_classification = {
                **classification,
                "selected_code": decision["override_code"],
                "reasoning": f"Human override: {decision.get('override_reason', 'N/A')}",
                "original_code": classification["selected_code"],
                "human_reviewed": True,
            }
            return {
                "classification": new_classification,
                "routing_decision": f"Human override → {decision['override_code']}",
            }
        else:
            # APPROVE: human confirmed the AI's decision
            return {
                "routing_decision": f"Human approved → {classification['selected_code']}",
            }

    # No review needed — auto-route
    return {
        "routing_decision": f"Auto-routed → {classification['selected_code']} (confidence {confidence:.2f})",
    }


print("Node 4 (review_and_route) defined — uses interrupt() for conditional pauses")

Node 4 (review_and_route) defined — uses interrupt() for conditional pauses


In [ ]:
# Global trainer log — in production this would be a database.
# For the demo, a list is enough to show the pattern.
trainer_log = []


def log_result(state: PipelineState) -> dict:
    """Node 5: Trainer pattern — log the final decision for future improvement.
    
    Captures the original AI prediction alongside any human corrections.
    Over time, these logs reveal patterns (e.g., "the model consistently
    misclassifies compound hazards") that inform prompt updates or fine-tuning.
    """
    classification = state["classification"]
    routing = state.get("routing_decision", "unknown")

    entry = {
        "timestamp": datetime.now().isoformat(),
        "selected_code": classification["selected_code"],
        "confidence": classification.get("confidence"),
        "reasoning": classification.get("reasoning"),
        "routing_decision": routing,
        "was_overridden": classification.get("original_code") is not None,
        "original_code": classification.get("original_code"),
    }
    trainer_log.append(entry)

    action = "OVERRIDE" if entry["was_overridden"] else "LOGGED"
    print(f"  [TRAINER LOG] {action}: {entry['selected_code']} (conf: {entry['confidence']:.2f})")
    if entry["was_overridden"]:
        print(f"               Original: {entry['original_code']} → Corrected: {entry['selected_code']}")

    return {"trainer_log": trainer_log.copy()}


print("Node 5 (log_result) defined — captures decisions for the trainer loop")

Node 5 (log_result) defined — captures decisions for the trainer loop


## Part 3: Wire the Graph with a Checkpointer

The **checkpointer** is what makes HITL possible in LangGraph. Without it, the pipeline runs start-to-finish with no way to pause. With `InMemorySaver`, the full state is saved at every node — so when `interrupt()` fires, we can inspect the state, make changes, and resume exactly where we left off.

Each pipeline run needs a **thread_id** — think of it as a session ID that ties the pause and resume together.

In [ ]:
# Build the graph: extract → retrieve → classify → review_and_route → log_result
workflow = StateGraph(PipelineState)

workflow.add_node("extract_entities", extract_entities)
workflow.add_node("retrieve_codes", retrieve_codes)
workflow.add_node("classify_problem", classify_problem)
workflow.add_node("review_and_route", review_and_route)
workflow.add_node("log_result", log_result)

workflow.add_edge(START, "extract_entities")
workflow.add_edge("extract_entities", "retrieve_codes")
workflow.add_edge("retrieve_codes", "classify_problem")
workflow.add_edge("classify_problem", "review_and_route")
workflow.add_edge("review_and_route", "log_result")
workflow.add_edge("log_result", END)

# InMemorySaver stores state at every node — required for interrupt() to work.
# In production you'd use a persistent checkpointer (e.g., SqliteSaver, PostgresSaver)
# so the state survives process restarts.
memory = InMemorySaver()
pipeline = workflow.compile(checkpointer=memory)

print("Pipeline compiled with checkpointer (HITL-enabled)")

Pipeline compiled with checkpointer (HITL-enabled)


## Part 4: Demo — Critical Severity Call (Validator Gate)

TX-001 is a pipe burst with water flooding — the model will classify it as Critical.

The pipeline will:
1. Extract entities → Retrieve codes → Classify (same as Week 3)
2. Hit `review_and_route` → see it's Critical → **pause via `interrupt()`**
3. You'll see the full dispatcher review screen and **type your decision live**: approve, override, or reject

In [ ]:
# Load transcripts
with open("transcripts.json") as f:
    transcripts = json.load(f)

# TX-001: Critical severity — pipe burst flooding the hallway
critical_transcript = transcripts[0]
print(f"Transcript: {critical_transcript['id']}")
print(f"Ground truth: {critical_transcript['true_category']}")
print(f"Text: {critical_transcript['transcript'][:120]}...")
print()

# Each run needs a unique thread_id — this ties the pause and resume together.
config = {"configurable": {"thread_id": "demo-critical-001"}}

# invoke() runs the pipeline until it either finishes or hits an interrupt.
# For this Critical call, it should pause at review_and_route.
print("Running pipeline...")
result = pipeline.invoke({"transcript": critical_transcript["transcript"]}, config)

print("\nPipeline paused — waiting for human review.")

Transcript: TX-001
Ground truth: PLUMB-001
Text: Hi, I'm calling from the third floor of the Westfield office building at 200 Main Street. There's water pouring from the...

Running pipeline...

  HUMAN REVIEW REQUIRED — severity is Critical
  Code:       PLUMB-001
  Confidence: 0.98
  Severity:   Critical
  Reasoning:  Ceiling pipe actively leaking and flooding hallway/carpet — directly matches 'Plumbing — Pipe Leak / Burst Pipe' (ceiling leak, soaked carpet, urgent response required). Severity critical supports immediate plumbing/flood response.

Pipeline paused — waiting for human review.


In [ ]:
# Inspect the pipeline state, then make your decision via the input prompt at the top of VS Code.
state = pipeline.get_state(config)
classification = state.values.get("classification", {})
entities = state.values.get("entities", {})
codes = state.values.get("retrieved_codes", [])

print("=" * 60)
print("  DISPATCHER REVIEW SCREEN")
print("=" * 60)
print()
print(f"  Severity:    {entities.get('severity')}")
print(f"  Problem:     {entities.get('problem_type')}")
print(f"  Location:    {entities.get('location_building')}, {entities.get('location_detail')}")
print(f"  Summary:     {entities.get('summary')}")
print()
print(f"  AI Classification:  {classification.get('selected_code')}")
print(f"  Confidence:         {classification.get('confidence')}")
print(f"  Reasoning:          {classification.get('reasoning')}")
print()
print("  Retrieved codes:")
for c in codes:
    print(f"    - {c['code']}: {c['content'][:60]}...")
print()
print("=" * 60)
print()

# --- Interactive human decision (prompt appears at top of VS Code window) ---
decision = input("Your decision — approve / override / reject: ").strip().lower()

if decision in ("approve", "y", "yes"):
    result = pipeline.invoke(Command(resume={"approved": True}), config)
    print(f"\n  APPROVED — routed as {classification.get('selected_code')}")
    print(f"  Routing: {result.get('routing_decision')}")

elif decision == "override":
    print("\nAvailable codes: PLUMB-001, PLUMB-002, ELEC-001, ELEC-002, HVAC-001,")
    print("  DOOR-001, DOOR-002, ELEV-001, JANI-001, JANI-002, SAFE-001, SAFE-002, ...")
    new_code = input("Enter correct code: ").strip().upper()
    reason = input("Reason for override: ").strip()
    result = pipeline.invoke(
        Command(resume={"override_code": new_code, "override_reason": reason}),
        config,
    )
    print(f"\n  OVERRIDDEN — {classification.get('selected_code')} → {new_code}")
    print(f"  Routing: {result.get('routing_decision')}")

else:
    print("\n  REJECTED — pipeline remains paused. Re-run this cell to review again.")

print(f"\n  Trainer log: {len(trainer_log)} entries")

Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]


  DISPATCHER REVIEW SCREEN

  Severity:    Severity.CRITICAL
  Problem:     Ceiling pipe leak causing water pouring into hallway and soaked carpet
  Location:    Westfield office building (200 Main Street), Third floor hallway near suite 310
  Summary:     On the third floor hallway near suite 310 at Westfield (200 Main Street) a ceiling pipe appears to be leaking, pouring water that has soaked the carpet and is spreading quickly — immediate response requested.

  AI Classification:  PLUMB-001
  Confidence:         0.98
  Reasoning:          Ceiling pipe actively leaking and flooding hallway/carpet — directly matches 'Plumbing — Pipe Leak / Burst Pipe' (ceiling leak, soaked carpet, urgent response required). Severity critical supports immediate plumbing/flood response.

  Retrieved codes:
    - PLUMB-001: PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking o...
    - PLUMB-001: PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking o...
    - PLUMB-001: PLUMB-001: Plumbing —

Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]



  HUMAN REVIEW REQUIRED — severity is Critical
  Code:       PLUMB-001
  Confidence: 0.98
  Severity:   Critical
  Reasoning:  Ceiling pipe actively leaking and flooding hallway/carpet — directly matches 'Plumbing — Pipe Leak / Burst Pipe' (ceiling leak, soaked carpet, urgent response required). Severity critical supports immediate plumbing/flood response.
  [TRAINER LOG] LOGGED: PLUMB-001 (conf: 0.98)

  APPROVED — routed as PLUMB-001
  Routing: Human approved → PLUMB-001

  Trainer log: 1 entries


In [ ]:
#**Key takeaway:** The pipeline paused, you saw all the context a dispatcher would see, and typed your decision. That's the **Validator** pattern — human approves *before* the action happens.

## Part 5: Low Severity Call — Auto-Routes (No Pause)

TX-008 is a minor janitorial issue. The model classifies it as Low severity with high confidence. The validator gate sees: not Critical, confidence > 0.7 → no interrupt, auto-route.

In [ ]:
# TX-008: Low severity janitorial issue — should auto-route with no interrupt.
low_transcript = transcripts[7]  # TX-008: pest/janitorial
print(f"Transcript: {low_transcript['id']}")
print(f"Ground truth: {low_transcript['true_category']}")
print(f"Text: {low_transcript['transcript'][:120]}...")
print()

# Different thread_id — each pipeline run needs its own thread.
config_low = {"configurable": {"thread_id": "demo-low-001"}}

print("Running pipeline...")
result = pipeline.invoke({"transcript": low_transcript["transcript"]}, config_low)

# This should complete without pausing — no interrupt fired.
print(f"\nROUTING: {result.get('routing_decision')}")
print(f"CLASSIFICATION: {result['classification']['selected_code']} (conf: {result['classification']['confidence']:.2f})")
print(f"MATCH: {'YES' if result['classification']['selected_code'] == low_transcript['true_category'] else 'NO'}")
print(f"\nNo interrupt — the call auto-routed because severity was not Critical and confidence was high.")

Transcript: TX-008
Ground truth: JANI-002
Text: Hello, I'm calling about the carpet in our office at suite 820, Madison Business Park. There's a stain in the main confe...

Running pipeline...
  [TRAINER LOG] LOGGED: JANI-002 (conf: 0.98)

ROUTING: Auto-routed → JANI-002 (confidence 0.98)
CLASSIFICATION: JANI-002 (conf: 0.98)
MATCH: YES

No interrupt — the call auto-routed because severity was not Critical and confidence was high.


## Part 6: Human Override — Changing the Classification

Now let's simulate the **Override Agent** pattern. The AI classifies a call — but this time it should genuinely get it **wrong**.

TX-011 is a strong chemical smell on the 8th floor. The keywords ("chemical smell", "nauseous", "evacuating", "gas leak") scream `SAFE-001` (Gas Leak / Chemical Hazard). But the caller also mentions the **floors were refinished over the weekend with polyurethane sealant** — the fumes are traveling up through the HVAC system. This is `HVAC-003` (Air Quality / Ventilation), not a gas leak. The fix is to increase fresh air intake and ventilate, not to call the gas company.

A human dispatcher with context would catch this. The AI can't — it just sees "chemical" + "evacuating" and panics.

In [ ]:
# TX-011: Chemical smell from floor refinishing fumes — AI will likely pick SAFE-001 (gas leak).
# Ground truth is HVAC-003 (air quality/ventilation — polyurethane fumes via HVAC).
tricky_transcript = transcripts[10]  # TX-011: chemical smell = refinishing fumes
print(f"Transcript: {tricky_transcript['id']}")
print(f"Ground truth: {tricky_transcript['true_category']}")
print(f"Text: {tricky_transcript['transcript'][:150]}...")
print()

config_override = {"configurable": {"thread_id": "demo-override-001"}}

print("Running pipeline...")
result = pipeline.invoke({"transcript": tricky_transcript["transcript"]}, config_override)

print("\nPipeline paused — let's see what the AI classified it as.")

Transcript: TX-007
Ground truth: SAFE-001
Text: We've got a serious situation at the parking garage at 500 Industrial Blvd. There's a strong smell of gas on the second ...

Running pipeline...

  HUMAN REVIEW REQUIRED — severity is Critical
  Code:       SAFE-001
  Confidence: 0.98
  Severity:   Critical
  Reasoning:  Reports indicate a strong gas odor on level 2 with Severity.CRITICAL and partial evacuation underway — matches ‘suspected gas leak’ and ‘evacuate’ keywords. SAFE-001 explicitly covers life-safety gas leaks/chemical hazards and requires emergency response.

Pipeline paused — let's see what the AI classified it as.


In [ ]:
# Check what the AI decided, then override via the input prompt at the top of VS Code.
state_ov = pipeline.get_state(config_override)
classification_ov = state_ov.values.get("classification", {})
entities_ov = state_ov.values.get("entities", {})
codes_ov = state_ov.values.get("retrieved_codes", [])

print("=" * 60)
print("  SUPERVISOR REVIEW — OVERRIDE DEMO")
print("=" * 60)
print()
print(f"  Severity:    {entities_ov.get('severity')}")
print(f"  Problem:     {entities_ov.get('problem_type')}")
print(f"  Location:    {entities_ov.get('location_building')}, {entities_ov.get('location_detail')}")
print(f"  Summary:     {entities_ov.get('summary')}")
print()
print(f"  AI Classification:  {classification_ov.get('selected_code')}")
print(f"  Confidence:         {classification_ov.get('confidence')}")
print(f"  Reasoning:          {classification_ov.get('reasoning')}")
print()
print("  Retrieved codes:")
for c in codes_ov:
    print(f"    - {c['code']}: {c['content'][:60]}...")
print()
print("=" * 60)
print()

decision = input("Your decision — approve / override: ").strip().lower()

if decision == "override":
    print("\nAvailable codes: PLUMB-001, PLUMB-002, ELEC-001, ELEC-002, HVAC-001,")
    print("  DOOR-001, DOOR-002, ELEV-001, JANI-001, JANI-002, SAFE-001, SAFE-002, ...")
    new_code = input("Enter correct code: ").strip().upper()
    reason = input("Reason for override: ").strip()
    result = pipeline.invoke(
        Command(resume={"override_code": new_code, "override_reason": reason}),
        config_override,
    )
    print(f"\n  OVERRIDDEN — {classification_ov.get('selected_code')} → {new_code}")
    print(f"  Reason: {reason}")
    print(f"  Routing: {result.get('routing_decision')}")
    print(f"  Original code: {result['classification'].get('original_code', 'N/A')}")
else:
    result = pipeline.invoke(Command(resume={"approved": True}), config_override)
    print(f"\n  APPROVED — routed as {classification_ov.get('selected_code')}")
    print(f"  Routing: {result.get('routing_decision')}")

print(f"\n  Trainer log: {len(trainer_log)} entries")

Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]


  SUPERVISOR REVIEW — OVERRIDE DEMO

  Severity:    Severity.CRITICAL
  Problem:     Strong smell of gas / possible gas leak
  Location:    Parking garage at 500 Industrial Blvd, Second level (Level 2) of the garage
  Summary:     Multiple reports of a strong gas odor on level 2 of the parking garage at 500 Industrial Blvd; partial evacuation underway and immediate inspection is required due to potential life-safety risk.

  AI Classification:  SAFE-001
  Confidence:         0.98
  Reasoning:          Reports indicate a strong gas odor on level 2 with Severity.CRITICAL and partial evacuation underway — matches ‘suspected gas leak’ and ‘evacuate’ keywords. SAFE-001 explicitly covers life-safety gas leaks/chemical hazards and requires emergency response.

  Retrieved codes:
    - SAFE-001: SAFE-001: Life Safety — Gas Leak / Chemical Hazard
Suspected...
    - SAFE-001: SAFE-001: Life Safety — Gas Leak / Chemical Hazard
Suspected...
    - SAFE-001: SAFE-001: Life Safety — Gas Leak / Chemic

Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]



  HUMAN REVIEW REQUIRED — severity is Critical
  Code:       SAFE-001
  Confidence: 0.98
  Severity:   Critical
  Reasoning:  Reports indicate a strong gas odor on level 2 with Severity.CRITICAL and partial evacuation underway — matches ‘suspected gas leak’ and ‘evacuate’ keywords. SAFE-001 explicitly covers life-safety gas leaks/chemical hazards and requires emergency response.
  [TRAINER LOG] OVERRIDE: PLUMB-001 (conf: 0.98)
               Original: SAFE-001 → Corrected: PLUMB-001

  OVERRIDDEN — SAFE-001 → PLUMB-001
  Reason: test
  Routing: Human override → PLUMB-001
  Original code: SAFE-001

  Trainer log: 3 entries


In [ ]:
#**Key takeaway:** Same `interrupt()` + `Command(resume=...)` mechanism, but you passed a different code. The original AI classification is preserved in the log — that's your training signal for the **Override Agent** pattern.

## Part 7: The Trainer Log — Learning from Human Decisions

Every decision flows through `log_result`. The trainer log captures both auto-routed calls and human-reviewed calls. Over time, this data reveals patterns:
- Which codes get overridden most?
- Does the model struggle with certain severity levels?
- Are there specific transcript patterns that lead to low confidence?

This is the **Trainer** pattern: async learning from human corrections.

In [ ]:
# Show the full trainer log — all 3 runs captured.
print(f"TRAINER LOG — {len(trainer_log)} entries\n")
print(f"{'#':<4} {'Code':<12} {'Conf':<6} {'Overridden?':<12} {'Original':<12} {'Routing'}")
print("-" * 80)

for i, entry in enumerate(trainer_log):
    override_str = "YES" if entry["was_overridden"] else "no"
    original = entry.get("original_code") or "—"
    print(f"{i+1:<4} {entry['selected_code']:<12} {entry['confidence']:.2f}  {override_str:<12} {original:<12} {entry['routing_decision']}")

print()

# Summary stats
overrides = sum(1 for e in trainer_log if e["was_overridden"])
auto_routed = sum(1 for e in trainer_log if "Auto-routed" in e.get("routing_decision", ""))
human_approved = sum(1 for e in trainer_log if "approved" in e.get("routing_decision", ""))

print(f"Auto-routed:     {auto_routed}")
print(f"Human approved:  {human_approved}")
print(f"Human overrides: {overrides}")
print()
print("In production, you'd analyze this weekly:")
print("  - Which codes get overridden? → Update those prompts")
print("  - What triggers low confidence? → Improve retrieval for those categories")
print("  - After 500+ entries → Consider fine-tuning")

TRAINER LOG — 3 entries

#    Code         Conf   Overridden?  Original     Routing
--------------------------------------------------------------------------------
1    PLUMB-001    0.98  no           —            Human approved → PLUMB-001
2    JANI-002     0.98  no           —            Auto-routed → JANI-002 (confidence 0.98)
3    PLUMB-001    0.98  YES          SAFE-001     Human override → PLUMB-001

Auto-routed:     1
Human approved:  1
Human overrides: 1

In production, you'd analyze this weekly:
  - Which codes get overridden? → Update those prompts
  - What triggers low confidence? → Improve retrieval for those categories
  - After 500+ entries → Consider fine-tuning


## Part 8: Bonus — Using `update_state()` Directly

`interrupt()` is the validator pattern (pause before action). But sometimes you want to change the state of a pipeline that **already completed** — the **post-execution override**. That's `update_state()`.

The low-severity call from Part 5 already auto-routed. Now you'll play the supervisor who reviews the queue after the fact — type a new code if you disagree.

In [ ]:
# The low-severity call already completed. Override it after the fact via input prompt.
state_before = pipeline.get_state(config_low)
current_code = state_before.values['classification']['selected_code']
current_routing = state_before.values.get('routing_decision', 'N/A')

print("=" * 60)
print("  POST-EXECUTION OVERRIDE — update_state()")
print("=" * 60)
print()
print(f"  This call already completed and was auto-routed.")
print(f"  Current code:    {current_code}")
print(f"  Current routing: {current_routing}")
print()

change = input("Override this classification? (yes/no): ").strip().lower()

if change in ("yes", "y"):
    print("\nAvailable codes: PLUMB-001, PLUMB-002, ELEC-001, ELEC-002, HVAC-001,")
    print("  DOOR-001, DOOR-002, ELEV-001, JANI-001, JANI-002, SAFE-001, SAFE-002, ...")
    new_code = input("Enter correct code: ").strip().upper()
    reason = input("Reason: ").strip()

    pipeline.update_state(
        config_low,
        {
            "classification": {
                **state_before.values["classification"],
                "selected_code": new_code,
                "reasoning": f"Supervisor override: {reason}",
                "original_code": current_code,
            },
            "routing_decision": f"Post-execution override → {new_code}",
        },
    )

    state_after = pipeline.get_state(config_low)
    print(f"\n  BEFORE: {current_code}")
    print(f"  AFTER:  {state_after.values['classification']['selected_code']}")
    print(f"  Checkpoint modified. In production, this triggers a re-routing notification.")
else:
    print("\n  No changes. Classification remains as-is.")

Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]


  POST-EXECUTION OVERRIDE — update_state()

  This call already completed and was auto-routed.
  Current code:    JANI-002
  Current routing: Auto-routed → JANI-002 (confidence 0.98)


Available codes: PLUMB-001, PLUMB-002, ELEC-001, ELEC-002, HVAC-001,
  DOOR-001, DOOR-002, ELEV-001, JANI-001, JANI-002, SAFE-001, SAFE-002, ...


Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]
Deserializing unregistered type __main__.Severity from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'Severity')]



  BEFORE: JANI-002
  AFTER:  JANI-001
  Checkpoint modified. In production, this triggers a re-routing notification.


In [ ]:
#**Key takeaway:** `update_state()` modifies the checkpoint directly — no need for `interrupt()`. This is useful when a supervisor reviews the routing queue *after* the pipeline already finished.

## Recap: The 3 HITL Patterns in This Notebook

| Pattern | What We Did | LangGraph Feature |
|---------|------------|-------------------|
| **Validator** | Paused on Critical / low confidence calls for human approval | `interrupt()` + `Command(resume=...)` |
| **Override** | Changed the classification after the AI decided (via resume or update_state) | `Command(resume={override_code: ...})` + `update_state()` |
| **Trainer** | Logged every decision (auto-routed, approved, overridden) for future improvement | `log_result` node writing to `trainer_log` |

## Assignment

See `Quiz, Assignment & Reading.pdf` for full details. In short:

1. **Add HITL checkpoints** to your agentic RAG system from Week 3
2. Implement a **validator gate**, an **override mechanism**, and a **trainer log**
3. Write a **half-page design document** explaining your choices

Think about: what condition triggers the validator? What information does the human need? How would this scale to 10,000 calls/day?